# Lecture 5 Lab — Unsupervised Anomaly Detection on Vibration Spectra
### Machine Learning for Robotics & Industrial Automation

| | |
|---|---|
| **Estimated time** | 3–4 hours |
| **Tools** | Python, NumPy, pandas, matplotlib, scikit-learn |
| **Submit** | This completed notebook (see §10) |

Type your name - surname and student ID in the cell below. Failure to do so resuls in -1 penalty for this lab.

In [ ]:
# your name - surname, student ID


### Important cell below 

You must assign 3 last digits of your student ID to this <code>s_id</code> variable. For example, if your student ID is 6710546988, you must assign
```python
s_id = 988
```
This code will be printed in later cells. 

Failure to do so results in -2 penalty. Further score deduction if the student ID is inconsitent throughout the notebook.

In [ ]:
s_id = ?  # replace with 3 last digits of your student ID.

## 1. Learning Objectives

By the end of this lab, you should be able to:

- Apply PCA to reduce a high-dimensional feature set and interpret explained variance.
- Visualize clustering results in reduced dimensions.
- Fit k-means and DBSCAN, and explain why they behave differently on the same data.
- Use DBSCAN's noise label to flag anomalies with no labeled examples of failure — and check your work against ground truth you were not allowed to train on.

## 2. Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans, DBSCAN

print("Environment OK")

## 3. Part A — Vibration Spectra Dataset

Each row is a 20-value frequency spectrum from one measurement cycle. Three operating states (`Idle`, `Normal`, `Heavy Load`) each produce a smooth, single-peak spectrum in a characteristic frequency range. A handful of injected anomalies simulate unusual fault signatures — a sharp peak somewhere unexpected, a double resonance, broadband energy, or an extreme-amplitude reading.

**`true_label` exists only so you can check your work at the end. Nothing in Parts C–F is allowed to look at it.**

In [ ]:
def generate_vibration_spectra(n_per_cluster=60, n_freq_bins=20, n_anomalies=15, seed=0):
    rng = np.random.default_rng(seed)
    cluster_specs = {
        "Idle":       dict(peak_bin=3, amplitude=1.0, width=2),
        "Normal":     dict(peak_bin=8, amplitude=2.0, width=2.5),
        "Heavy Load": dict(peak_bin=14, amplitude=3.0, width=3),
    }
    freq = np.arange(n_freq_bins)
    rows, labels = [], []
    for cname, spec in cluster_specs.items():
        for _ in range(n_per_cluster):
            center = spec["peak_bin"] + rng.normal(0, 0.7)
            amp = spec["amplitude"] + rng.normal(0, 0.2)
            spectrum = amp * np.exp(-0.5 * ((freq - center) / spec["width"]) ** 2)
            spectrum += rng.normal(0, 0.08, n_freq_bins)
            rows.append(spectrum); labels.append(cname)

    # Diverse anomalies -- each one a different kind of "weird" spectrum, so
    # they scatter in feature space rather than forming their own tight cluster.
    for _ in range(n_anomalies):
        kind = rng.integers(0, 4)
        if kind == 0:  # sharp peak at an unusual frequency
            center = rng.uniform(0, n_freq_bins)
            spectrum = rng.uniform(2.5, 4.5) * np.exp(-0.5 * ((freq - center) / 1.0) ** 2)
        elif kind == 1:  # double peak (two resonances at once)
            c1, c2 = rng.uniform(0, n_freq_bins, 2)
            spectrum = 2.0 * np.exp(-0.5 * ((freq - c1) / 1.5) ** 2) + 2.0 * np.exp(-0.5 * ((freq - c2) / 1.5) ** 2)
        elif kind == 2:  # broadband / flat energy
            spectrum = np.full(n_freq_bins, rng.uniform(1.0, 2.0))
        else:  # extreme-amplitude peak
            center = rng.uniform(4, 12)
            spectrum = rng.uniform(5.0, 7.0) * np.exp(-0.5 * ((freq - center) / 2.0) ** 2)
        spectrum = spectrum + rng.normal(0, 0.15, n_freq_bins)
        rows.append(spectrum); labels.append("Anomaly")

    X = np.array(rows)
    df = pd.DataFrame(X, columns=[f"bin_{i}" for i in range(n_freq_bins)])
    df["true_label"] = labels
    return df.sample(frac=1, random_state=seed).reset_index(drop=True)


df = generate_vibration_spectra()
feat_cols = [c for c in df.columns if c.startswith("bin_")]
print(df.shape)
df.head()

In [ ]:
print("Student ID : "+str(s_id))
# A quick look at a few raw spectra, colored by their (normally hidden) true label
fig, ax = plt.subplots(figsize=(7, 4))
colors = {"Idle": "#3E5C76", "Normal": "#FF6A39", "Heavy Load": "#1B2A41", "Anomaly": "#6B4F9C"}
for label in ["Idle", "Normal", "Heavy Load", "Anomaly"]:
    sample = df[df["true_label"] == label].iloc[0][feat_cols].values
    ax.plot(sample, color=colors[label], label=label, linewidth=2 if label != "Anomaly" else 1.5,
            linestyle="--" if label == "Anomaly" else "-")
ax.set_xlabel("Frequency bin"); ax.set_ylabel("Amplitude"); ax.legend()
ax.set_title("One example spectrum per (hidden) true label")
plt.tight_layout()
plt.show()

## 4. Part B — Why Not Cluster Directly in 20 Dimensions?

Before reducing dimensions, take a look at how spread out the raw features are. There isn't a TODO here — just run the cell and look at the result.

In [ ]:
print("Student ID : "+str(s_id))
X = df[feat_cols].values
print(f"Raw feature matrix: {X.shape[0]} rows x {X.shape[1]} columns")
print("Per-bin standard deviation (how much each frequency bin varies across all readings):")
print(pd.Series(X.std(axis=0), index=feat_cols).round(2).to_string())

Notice how correlated neighboring bins are — a peak at bin 8 also raises bins 7 and 9. That redundancy is exactly what PCA is built to compress away.

## 5. Part C — Reduce to 2 Components with PCA

Fill in the `TODO` below.

In [ ]:
# TODO : standardize the data. From X create X_std using StandardScaler()

X_std = None


In [ ]:
print("Student ID : "+str(s_id))
# TODO: create a PCA object that reduces to 2 components, fit it on X,
# and transform X into `pca_coords`.
pca = None
pca_coords = None


print("Explained variance ratio:", np.round(pca.explained_variance_ratio_, 3))
print(f"Total variance captured by 2 components: {pca.explained_variance_ratio_.sum():.1%}")

In [ ]:
# Sanity-check plot ONLY -- colored by true_label, which you are not allowed
# to use in Parts E and F. This is just to build your intuition about the space.
print("Student ID : "+str(s_id))
plt.figure(figsize=(6.5, 5))
for label in ["Idle", "Normal", "Heavy Load", "Anomaly"]:
    mask = df["true_label"] == label
    plt.scatter(pca_coords[mask, 0], pca_coords[mask, 1], s=25, color=colors[label], label=label,
                marker="x" if label == "Anomaly" else "o")
plt.xlabel("PC1"); plt.ylabel("PC2"); plt.legend()
plt.title("PCA-reduced spectra (colored by hidden true label, for reference only)")
plt.tight_layout()
plt.show()

## 6. Part D — Cluster with k-Means

You know from the lecture that three operating states exist, so `k=3` is a reasonable choice here. Fit k-means **on `pca_coords`**, not on the raw 20-dimensional data.

In [ ]:
# TODO : use K-means algorithm with k = 3 and fit on pca_coord

kmeans = None
kmeans_label = None

print("Student ID : "+str(s_id))
plt.figure(figsize=(6.5, 5))
plt.scatter(pca_coords[:, 0], pca_coords[:, 1], c=kmeans_labels, cmap="viridis", s=25)
plt.scatter(kmeans.cluster_centers_[:, 0], kmeans.cluster_centers_[:, 1],
            c="red", marker="X", s=150, edgecolor="black", label="centroids")
plt.xlabel("PC1"); plt.ylabel("PC2"); plt.legend()
plt.title("k-means (k=3) on PCA-reduced spectra")
plt.tight_layout()
plt.show()

In [ ]:
# Now, and ONLY now, compare against the hidden true label to check your work.
print("Student ID : "+str(s_id))
print(pd.crosstab(df["true_label"], kmeans_labels, rownames=["true label"], colnames=["k-means cluster"]))

Look closely at the `Anomaly` row: k-means had to put every single anomaly into one of the three clusters. There is no "none of the above" option — that's the limitation to keep in mind for Part F.

## 7. Part E — Cluster with DBSCAN

Fill in the `TODO` below. Start with `eps=0.8` and `min_samples=5`, then feel free to experiment — small changes to `eps` noticeably change how much gets flagged as noise.

In [ ]:
# TODO: fit DBSCAN on pca_coords with eps=0.8 and min_samples=5, and store
# the cluster labels (noise points are labeled -1) in `dbscan_labels`.
dbscan = None
dbscan_labels = None

print("Student ID : "+str(s_id))
n_clusters = len(set(dbscan_labels)) - (1 if -1 in dbscan_labels else 0)
n_noise = list(dbscan_labels).count(-1)
print(f"DBSCAN found {n_clusters} clusters and flagged {n_noise} points as noise/anomalies")

In [ ]:
print("Student ID : "+str(s_id))
plt.figure(figsize=(6.5, 5))
is_noise = dbscan_labels == -1
plt.scatter(pca_coords[~is_noise, 0], pca_coords[~is_noise, 1], c=dbscan_labels[~is_noise], cmap="viridis", s=25, label="clustered")
plt.scatter(pca_coords[is_noise, 0], pca_coords[is_noise, 1], facecolors="none", edgecolors="black", s=80, linewidths=1.5, label="flagged as noise")
plt.xlabel("PC1"); plt.ylabel("PC2"); plt.legend()
plt.title("DBSCAN on PCA-reduced spectra")
plt.tight_layout()
plt.show()

## 8. Part F — Did DBSCAN Actually Catch the Anomalies?

Now check your work against the hidden true label.

In [ ]:
print("Student ID : "+str(s_id))
print(pd.crosstab(df["true_label"], dbscan_labels, rownames=["true label"], colnames=["DBSCAN label (-1 = noise)"]))
print()

anomaly_mask = (df["true_label"] == "Anomaly").values
caught = (is_noise & anomaly_mask).sum()
missed = (~is_noise & anomaly_mask).sum()
false_flags = (is_noise & ~anomaly_mask).sum()
print(f"True anomalies correctly flagged as noise : {caught} / {anomaly_mask.sum()}")
print(f"True anomalies missed (assigned a cluster) : {missed} / {anomaly_mask.sum()}")
print(f"Normal points incorrectly flagged as noise  : {false_flags}")

## 9. Reflection Questions

Answer briefly (2–3 sentences each) by editing the markdown cells below.

**1. How many principal components would you need to capture, say, 95% of the variance? What does the answer suggest about the "effective dimensionality" of this vibration data?**

*Your answer:*

**2. Look at your k-means crosstab from §6. Where did k-means get "confused," and why — given that it has no way to say "none of the above"?**

*Your answer:*

**3. How did DBSCAN's catch rate look in §8? Try lowering `eps` slightly and re-running §7–§8 — what happens to the number of caught anomalies versus the number of false flags?**

*Your answer:*

**4. `true_label` never appeared in Parts C–F until the very end. Why is that essential for this to count as a legitimate test of unsupervised anomaly detection, rather than something you accidentally tuned to the answer key?**

*Your answer:*

## 10. Deliverables & Submission

- This notebook, completed and able to run top-to-bottom without errors (`Kernel → Restart & Run All`).
- The PCA scatter plot from §5, the k-means plot and crosstab from §6, and the DBSCAN plot and crosstab from §7–§8.
- Your written answers to the four reflection questions.

Submit this `.ipynb` file in google classroom before the due date. Late penalty is -1 per day.

## 11. Grading Rubric Guide

| Component | Weight |
|---|---|
| PCA correctly fit and explained variance interpreted | 20% |
| k-means correctly fit and results interpreted | 20% |
| DBSCAN correctly fit and anomaly catch rate evaluated | 30% |
| Reflection questions | 20% |
| Notebook quality (runs cleanly top-to-bottom, reasonably organized) | 10% |

---
**Next week:** *Model Evaluation & Hyperparameter Tuning* — cross-validation, grid search, and pipelines, tying the whole semester's classical ML together.

Generated by Claude and customized by

<div align="center">
<img src="https://raw.githubusercontent.com/dewdotninja/sharing-github/refs/heads/master/dewninja_logo50.jpg" alt="dewninja"/>
</div>
<div align="center">dew.ninja 2026</div>
